In [ ]:
# Install the GitHub-pinned native library and profile registry.
from pathlib import Path
from urllib.request import urlretrieve
from runpy import run_path
from tempfile import TemporaryDirectory
import json
import os

SETUP_COMMIT = "aa77ec7665ef98a8030c5c449c0bc95d04028787"
setup_url = (
    "https://raw.githubusercontent.com/TryDotAtwo/MultiGPUBeamSearch/"
    f"{SETUP_COMMIT}/kaggle/cube555_github/setup.py"
)
with TemporaryDirectory() as setup_dir:
    setup_file, _ = urlretrieve(setup_url, setup_dir + "/setup.py")
    run_path(setup_file)

SOLVER_COMMIT = os.environ["MULTIGPUBEAMSEARCH_SOURCE_COMMIT"]
WORK_ROOT = Path(
    os.environ.get(
        "MULTIGPUBEAMSEARCH_HOME",
        Path.home() / ".cache/multigpubeamsearch/cube555",
    )
)
source_notebook = (
    WORK_ROOT
    / "sources"
    / SOLVER_COMMIT
    / "kaggle/cube555_github/cube555-native-2xt4-blend.ipynb"
)
NOTEBOOK_PATH = WORK_ROOT / "publication-notebook.ipynb"
notebook = json.loads(source_notebook.read_text(encoding="utf-8"))
notebook["cells"][0]["source"] = [
    line.replace("__SETUP_COMMIT__", SETUP_COMMIT)
    for line in notebook["cells"][0]["source"]
]
NOTEBOOK_PATH.write_text(json.dumps(notebook, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")


In [ ]:
import os

# Puzzle and input
INPUT_BUNDLE_URL = "https://github.com/TryDotAtwo/MultiGPUBeamSearch/releases/download/cube555-inputs-20261001/cube555-inputs.zip"
INPUT_CACHE_DIR = "~/.cache/multigpubeamsearch/cube555"
PUZZLE_ID_START, PUZZLE_ID_END = 1020, 1020
INDICES_URL = None
RANDOM_PUZZLE = False
STATE_ID_COLUMN, STATE_COLUMN = "initial_state_id", "initial_state"

# Model files and blend: weight 1.0 = Transformer, 0.0 = MLP
TRANSFORMER_CHECKPOINT = "q555_f1_bell2k.pt"
MLP_CHECKPOINT = "q555_2k_BEST.pt"
LAYOUT_FILE = "piece_layout_555.json"
TRANSFORMER_WEIGHT = 0.8

# Search
BEAM_WIDTH = 1_048_576
MAX_STEPS = 140
BEAM_MODE = "simple"
USE_CHILD_SCORES = True
RETURN_PATH = True
NUM_GPUS = None  # All visible GPUs

# Collect, BFS and reflection
SOLUTION_MODE = "collect"
COLLECT_EXTRA_DEPTHS = 1
MAX_COLLECTED_SOLUTIONS = 2_000
TOUCH_BFS_RADIUS = 5
TOUCH_BFS_MAX_ENTRIES = 30_000_000
REFLECT_MODE = "off"
REFLECT_SOURCE_CSV = ""
REFLECT_ID_COLUMN, REFLECT_PATH_COLUMN = "puzzle_id", "solution"

# Output and publication
OUTPUT_ROOT = "beam_results"
PUBLISH_RESULTS = True  # Set False outside Kaggle
PUBLISH_MODEL_METADATA = None
COMPETITION, PUZZLE_TYPE = "cayley-py-555-cube", "cube_5/5/5"
AUTHOR, KAGGLE_OWNER = "trydotatwo", "trydotatwo"
KAGGLE_SLUG = "cube555-native-2xt4-blend"
KAGGLE_VERSION = int(os.environ.get("KAGGLE_KERNEL_VERSION", "181"))


In [ ]:
import json
import sys
import tempfile
from pathlib import Path

import cayleypy as cp
import multigpubeamsearch as beam
from multigpubeamsearch.contracts import GraphContract

# Read the Cube555 graph and puzzle states.
inputs = beam.download_bundle(INPUT_BUNDLE_URL, INPUT_CACHE_DIR)
model_root = inputs / "model"
graph_info_path = model_root / "puzzle_info.json"
graph_info = json.loads(graph_info_path.read_text(encoding="utf-8"))
generators = graph_info["generators"]
graph = cp.CayleyGraph(
    cp.CayleyGraphDef.create(
        list(generators.values()),
        generator_names=list(generators),
        central_state=graph_info["central_state"],
    ),
    device="cpu",
    random_seed=42,
)
start_states = beam.read_states(
    inputs / "competition" / "test.csv",
    id_column=STATE_ID_COLUMN,
    state_column=STATE_COLUMN,
)
puzzle_ids = beam.select_ids(
    PUZZLE_ID_START, PUZZLE_ID_END,
    indices_url=INDICES_URL,
    random_choice=RANDOM_PUZZLE,
)

# Export and blend the two native Cube555 models.
source_root = Path(beam.__file__).resolve().parent / "_native_source"
sys.path.insert(0, str(source_root))
from tools.cube555.export import export_native_models

export_root = Path(INPUT_CACHE_DIR).expanduser() / "exports"
export_root.mkdir(parents=True, exist_ok=True)
model_output = Path(tempfile.mkdtemp(prefix="cube555-", dir=export_root)) / "models"
models = export_native_models(
    model_root / TRANSFORMER_CHECKPOINT,
    model_root / MLP_CHECKPOINT,
    model_root / LAYOUT_FILE,
    graph_info_path,
    model_output,
    contract=GraphContract.from_graph(graph, graph.definition.central_state),
)
predictor = beam.NativeEnsemble(models, (TRANSFORMER_WEIGHT, 1.0 - TRANSFORMER_WEIGHT))

reflection_paths = {}
if REFLECT_SOURCE_CSV:
    reflection_paths = beam.read_paths(
        REFLECT_SOURCE_CSV, graph,
        id_column=REFLECT_ID_COLUMN,
        path_column=REFLECT_PATH_COLUMN,
    )

beam.enable_native(beam.NativeOptions(num_gpus=NUM_GPUS))


In [ ]:
from pathlib import Path

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

# Keep models and GPU buffers alive while processing puzzle IDs.
with beam.NativeSession():
    for puzzle_id in puzzle_ids:
        start_state = start_states[puzzle_id]
        result = graph.beam_search(
            start_state=start_state,
            predictor=predictor,
            backend="native",
            beam_width=BEAM_WIDTH,
            max_steps=MAX_STEPS,
            beam_mode=BEAM_MODE,
            use_child_scores=USE_CHILD_SCORES,
            return_path=RETURN_PATH,
            solution_mode=SOLUTION_MODE,
            collect_extra_depths=COLLECT_EXTRA_DEPTHS,
            max_collected_solutions=MAX_COLLECTED_SOLUTIONS,
            bfs_radius=TOUCH_BFS_RADIUS,
            bfs_max_entries=TOUCH_BFS_MAX_ENTRIES,
            reflect_mode=REFLECT_MODE,
            reflection_source=reflection_paths.get(str(puzzle_id)),
        )
        saved = beam.save_result(
            result, graph, start_state, OUTPUT_ROOT,
            metadata={"problem_id": str(puzzle_id)},
        )
        print(f"Puzzle {puzzle_id}: best length {result.path_length}; "
              f"solutions {len(result.solutions)}; saved {saved}")

        if PUBLISH_RESULTS:
            publication = beam.publish_result(
                result, graph, start_state,
                puzzle_id=puzzle_id,
                predictor=predictor,
                author=AUTHOR,
                kaggle_owner=KAGGLE_OWNER,
                kaggle_slug=KAGGLE_SLUG,
                kaggle_version=KAGGLE_VERSION,
                notebook_path=NOTEBOOK_PATH,
                solver_commit=SOLVER_COMMIT,
                competition=COMPETITION,
                puzzle_type=PUZZLE_TYPE,
                model_metadata=PUBLISH_MODEL_METADATA,
                directory=OUTPUT_ROOT,
            )
            print("Publication:", publication)
